# Dataset downloads, one source at a time

These cells validate and reuse completed downloads. They do not start training or overwrite the frozen main benchmark. A failed source can be retried separately. See SUBMISSION_HANDOFF.md for provenance, terms and optional larger data.

In [1]:
import os, sys, json
from pathlib import Path
start = Path(os.environ.get("TOXICITY_PROJECT_ROOT", Path.cwd())).resolve()
ROOT = next(p for p in (start, *start.parents) if (p / "contextual/config.py").exists())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from contextual.config import load_config
from contextual.external_data import fetch_sources, prepare_external
settings = json.loads((ROOT / "configs/data_sources.json").read_text())

## CGA-WIKI: original benchmark source

Only download/extract here. Keep the existing prepared data unchanged for model comparison.

In [2]:
from contextual.data import download
download(load_config(ROOT / "configs/contextual_submission.json"))

Using verified local archive: D:\NLP_projects\Context_Aware_Toxicity_classifier\data\raw\contextual\cga-wiki.zip


WindowsPath('D:/NLP_projects/Context_Aware_Toxicity_classifier/data/raw/contextual/cga-wiki')

## ToxiChat: English human conversation turns

Model-generated continuations are excluded. The original train/dev/test assignment is retained as provenance, then shared-text groups receive new splits for this separate experiment.

In [3]:
fetch_sources({**settings, "sources": [settings["sources"][0]], "download_manifest": "reports/toxichat_downloads.json"})

Verified: train.jsonl
Verified: dev.jsonl
Verified: test.jsonl


[{'source': 'toxichat_human', 'path': 'D:\\NLP_projects\\Context_Aware_Toxicity_classifier\\data\\raw\\external\\toxichat_human\\train.jsonl', 'sha256': '806da95ba491888626a18c0324e9d6ec906450ba9fef92b54343ebf44e48682a'}, {'source': 'toxichat_human', 'path': 'D:\\NLP_projects\\Context_Aware_Toxicity_classifier\\data\\raw\\external\\toxichat_human\\dev.jsonl', 'sha256': '591742cdf73b9fcae4f555cbccb5d49237b470172dd0fd07a4829cebc0060ace'}, {'source': 'toxichat_human', 'path': 'D:\\NLP_projects\\Context_Aware_Toxicity_classifier\\data\\raw\\external\\toxichat_human\\test.jsonl', 'sha256': '13e67afb0806b1865d446ea3d730578d750d9be4127fdd872ef7cc903b52f9bb'}]

## MACD: Telugu isolated comments

Source label 0 means abusive, and 1 means non-abusive. The adapter converts these to this project's toxic=1 and safe=0. These examples do not supply conversation context.

In [4]:
fetch_sources({**settings, "sources": [settings["sources"][1]], "download_manifest": "reports/macd_downloads.json"})

Verified: telugu_train.csv
Verified: telugu_val.csv
Verified: telugu_test.csv


[{'source': 'macd_telugu', 'path': 'D:\\NLP_projects\\Context_Aware_Toxicity_classifier\\data\\raw\\external\\macd_telugu\\telugu_train.csv', 'sha256': 'cb6bb4e4574cdad1477aa8c2c030c133ecc8ec1b5411c73b98370cb599aa2a69'}, {'source': 'macd_telugu', 'path': 'D:\\NLP_projects\\Context_Aware_Toxicity_classifier\\data\\raw\\external\\macd_telugu\\telugu_val.csv', 'sha256': '1553fbe42b32b66f2c535f14d2f763464cf54e1e06dfafc637612fe616429a18'}, {'source': 'macd_telugu', 'path': 'D:\\NLP_projects\\Context_Aware_Toxicity_classifier\\data\\raw\\external\\macd_telugu\\telugu_test.csv', 'sha256': '626d4907c0282ec9ead25b5d0e1ea664dffaeeab9744d55c34f107e8cd931e5c'}]

## Validate and prepare the separate external experiment

Conflicting labels for identical isolated texts are quarantined. All exact normalized message overlaps and conversations are kept together. Existing main benchmark and checkpoints are preserved.

In [5]:
external_manifest = prepare_external(settings)
external_manifest

{'examples': 34556, 'contextual_examples': 2623, 'sources': {'toxichat_human': 4560, 'macd_telugu': 29996}, 'splits': {'test': 5168, 'train': 24275, 'val': 5113}, 'quarantined_conflicting_isolated_rows': 67, 'prepared_sha256': 'bec6f7cd2b16e2ceb9b8b83a7ebc099367bd4a3a65c5b1b7f58f18f7a04dc131', 'policy': 'New shared-text/conversation grouped splits; not official benchmark splits. Original split is retained as provenance. External experiment only; frozen main benchmark unchanged.'}

## Continue

Run `python -m contextual.experiments --config configs/contextual_external.json` for its independent TF-IDF baseline. After GPU recovery run `python -m contextual.adapter_finetune --config configs/contextual_external.json`. Do not compare these scores directly with scores on the original benchmark. CivilComments is optional and was inaccessible (HTTP 403); detailed steps are in SUBMISSION_HANDOFF.md.